In [81]:
!pip install numpy pandas


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


# Imports

In [82]:
import json
import numpy as np
import pandas as pd
from pathlib import Path

# Settings 

In [83]:
ROOT          = Path.cwd().parent.parent
RAW_DIR       = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

GAME_TYPES = {
    "V75": "V85",   # ATG replaced V75 with V85 in Oct 2025
    "V85": "V85",
    "V64": "V64",
    "V86": "V86",
}

# Clean up data / Help functions

In [84]:
def load(path):
    return json.loads(path.read_text(encoding="utf-8"))

def total_race_time(t, distance_m):
    if not t or not distance_m:
        return None
    seconds_per_km = t.get("minutes", 0) * 60 + t.get("seconds", 0) + t.get("tenths", 0) / 10
    return seconds_per_km * (distance_m / 1000)

CHAR_MAP = str.maketrans("åäæÅÄÆøöØÖ", "aaaAAAooOO")

def normalize_name(name):
    return name.translate(CHAR_MAP) if name else name

# Extract starters

In [85]:
def extract_starters(date_str, calendar):
    rows = []

    for game_type, game_list in calendar.get("games", {}).items():
        if game_type not in GAME_TYPES:
            continue

        for game_stub in game_list:
            game_id   = game_stub["id"]
            game_path = RAW_DIR / f"game_{game_id}.json"

            if not game_path.exists():
                continue

            game = load(game_path)

            for race in game.get("races", []):
                race_id   = race.get("id")
                race_path = RAW_DIR / f"race_{race_id}.json"
                race_data = load(race_path) if race_path.exists() else race

                distance_m = race_data.get("distance")

                for start in race_data.get("starts", []):
                    horse     = start.get("horse", {})
                    driver    = start.get("driver", {})
                    trainer   = horse.get("trainer", {})
                    result    = start.get("result", {}) or {}
                    scratched = start.get("scratched", False)

                    rows.append({
                        "date":            date_str,
                        "game_type":       game_type,
                        "game_type_norm":  GAME_TYPES[game_type],
                        "game_id":         game_id,
                        "race_id":         race_id,
                        "race_name":       race_data.get("name"),
                        "track":           (race_data.get("track") or {}).get("name"),
                        "division":        race_data.get("number"),
                        "distance_m":      distance_m,
                        "start_method":    race_data.get("startMethod"),
                        "start_number":    start.get("number"),
                        "post_position":   start.get("postPosition"),
                        "scratched":       scratched,
                        "horse_id":        horse.get("id"),
                        "horse_name":      normalize_name(horse.get("name")),
                        "horse_age":       horse.get("age"),
                        "horse_sex":       horse.get("sex"),
                        "driver_id":       driver.get("id"),
                        "driver_name":     normalize_name(driver.get("firstName", "") + " " + driver.get("lastName", "")),
                        "trainer_id":      trainer.get("id"),
                        "trainer_name":    normalize_name(trainer.get("firstName", "") + " " + trainer.get("lastName", "")),
                        "finish_position": np.nan if scratched else result.get("finishOrder"),
                        "odds":            np.nan if scratched else result.get("finalOdds"),
                        "race_time_sec":   np.nan if scratched else total_race_time(result.get("kmTime"), distance_m),
                        "prize_money":     np.nan if scratched else result.get("prizeMoney"),
                        "won":             0    if scratched else int(result.get("finishOrder") == 1),
                    })
    return rows

# Build starters table

In [86]:
DONE_FILE = PROCESSED_DIR / "done_dates.txt"

done = set(DONE_FILE.read_text().splitlines()) if DONE_FILE.exists() else set()

all_rows = []

for cal_file in sorted(RAW_DIR.glob("calendar_*.json")):
    date_str = cal_file.stem.replace("calendar_", "")

    if date_str in done:
        print(f"Skipping {date_str} (already processed)")
        continue

    all_rows.extend(extract_starters(date_str, load(cal_file)))

    with DONE_FILE.open("a") as f:
        f.write(date_str + "\n")

df = pd.DataFrame(all_rows)

Skipping 2026-02-08 (already processed)
Skipping 2026-02-09 (already processed)
Skipping 2026-02-10 (already processed)
Skipping 2026-02-11 (already processed)
Skipping 2026-02-12 (already processed)
Skipping 2026-02-13 (already processed)
Skipping 2026-02-14 (already processed)
Skipping 2026-02-15 (already processed)
Skipping 2026-02-16 (already processed)
Skipping 2026-02-17 (already processed)
Skipping 2026-02-18 (already processed)
Skipping 2026-02-19 (already processed)
Skipping 2026-02-20 (already processed)
Skipping 2026-02-21 (already processed)
Skipping 2026-02-22 (already processed)
Skipping 2026-02-23 (already processed)
Skipping 2026-02-24 (already processed)
Skipping 2026-02-25 (already processed)
Skipping 2026-02-26 (already processed)
Skipping 2026-02-27 (already processed)
Skipping 2026-02-28 (already processed)
Skipping 2026-03-01 (already processed)
Skipping 2026-03-02 (already processed)
Skipping 2026-03-03 (already processed)
Skipping 2026-03-04 (already processed)


# Clean types

In [87]:
df["date"]             = pd.to_datetime(df["date"])
df["distance_m"]       = pd.to_numeric(df["distance_m"],       errors="coerce")
df["odds"]             = pd.to_numeric(df["odds"],             errors="coerce")
df["finish_position"]  = pd.to_numeric(df["finish_position"],  errors="coerce")
df["horse_age"]        = pd.to_numeric(df["horse_age"],        errors="coerce")
df["scratched"]        = df["scratched"].astype(bool)

for col in ["horse_id", "driver_id"]:
    df[col] = df[col].replace(0, np.nan)

for col in ["finish_position", "odds", "race_time_sec"]:
    df.loc[df[col] == 0, col] = np.nan

# Build horse and driver tables

In [88]:
def build_stats(df, key_col, name_col):
    active = df[~df["scratched"]].dropna(subset=[key_col])
    stats  = active.groupby(key_col).agg(
        name           = (name_col,         "first"),
        starts         = ("won",            "count"),
        wins           = ("won",            "sum"),
        avg_odds       = ("odds",           "mean"),
        avg_position   = ("finish_position","mean"),
        first_start    = ("date",           "min"),
        last_start     = ("date",           "max"),
    ).reset_index()
    stats["win_pct"] = (stats["wins"] / stats["starts"] * 100).round(1)
    return stats.sort_values("starts", ascending=False)

df_horses  = build_stats(df, "horse_id",  "horse_name")
df_drivers = build_stats(df, "driver_id", "driver_name")

# Save to CSV

In [89]:
for filename, table in {
    "starters.csv": df,
    "horses.csv":   df_horses,
    "drivers.csv":  df_drivers,
}.items():
    path = PROCESSED_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8")
    print(f"✓ {filename:<20} {len(table):>7,} rows   {path.stat().st_size // 1024} KB")

✓ starters.csv           3,528 rows   751 KB
✓ horses.csv             1,934 rows   121 KB
✓ drivers.csv              477 rows   34 KB


In [90]:
# ── Quick data quality check ──────────────────────────────────
print(f"Total rows:    {len(df):,}")
print(f"Date range:    {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Scratched:     {df['scratched'].sum():,} ({df['scratched'].mean()*100:.1f}%)")
print()

print("NaN per column:")
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0].to_string())
print()

print("Rows per game type:")
print(df["game_type"].value_counts().to_string())
print()

print("Rows per track:")
print(df["track"].value_counts().to_string())

Total rows:    3,528
Date range:    2025-12-22 → 2026-05-19
Scratched:     310 (8.8%)

NaN per column:
race_name          1302
post_position         6
horse_id            424
driver_id           447
trainer_id          440
finish_position     407
odds                310
race_time_sec       708
prize_money         534

Rows per game type:
game_type
V64    1719
V85    1352
V86     457

Rows per track:
track
Solvalla      417
Romme         363
Gävle         251
Örebro        249
Axevalla      247
Eskilstuna    233
Boden         206
Jägersro      201
Färjestad     171
Umåker        166
Kalmar        164
Bjerke        140
Momarken       94
Åby            93
Skive          91
Ålborg         71
Bollnäs        71
Halmstad       71
Skellefteå     67
Jarlsberg      63
Östersund      58
Bergsåker      41
